# Nagelhus et al. (2023), decoding with subsets of cells

**Paper.** Nagelhus et al. (2023), *Neuron* 111, 2091-2104,
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008); claims file written
from the version of record with its STAR Methods and Supplemental information.

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), version v1, the version
whose DOI the paper cites and the only one published (data check in `01_single_cells.ipynb`).

| Claim | Tested or blocked |
|---|---|
| fig3d-number-of-cells | tested |
| fig6c-subsets-decline | tested |
| fig6d-reorganization-grows-with-cells | tested |
| figs8-smaller-datasets | tested |

Random subsets of cells are drawn with a permutation of the cells, the first k kept (Methods,
Subsampling for decoder). Every decoder runs under both speed readings and both rate-map readings.
The Neuropixels sessions' tracking comes from the CA1 file (y is a copy of x in the population
files; notebook 01).

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)

started = time.time()
RAW_POP = []
for p in oc.POP_FILES:
    RAW_POP += oc.load(p)[0]
CA1, _ = oc.load(oc.SINGLE_FILES["CA1"])
POP, replaced = oc.decoding_sessions(RAW_POP, CA1)
del CA1
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION}: read in {time.time() - started:.0f} s; tracking from the CA1 file for {replaced}")
READINGS = [(sp, rm) for sp in "ab" for rm in "b"]
PARTS = {(k, sp): (oc.half_parts(p.trials[0], sp == "a"), oc.half_parts(p.trials[1], sp == "a")) for k, p in enumerate(POP) for sp in "ab"}
CACHE = {}


def decode(k, sp, rm, units, mode):
    pA, pB = PARTS[k, sp]
    return oc.decode_pair(pA, pB, mode, obj=POP[k].trials[1].obj, rm=rm, units=units, cache=CACHE.setdefault((k, sp), {}))["acc"]


BIG = [k for k, p in enumerate(POP) if p.n_units == 620][0]
MID = [k for k, p in enumerate(POP) if p.n_units == 135][0]
TET = [k for k, p in enumerate(POP) if p.technique == "tetrodes"]

In [ ]:
print(pd.DataFrame([{"session": f"{p.rat}#{p.index}", "technique": p.technique, "cells": p.n_units,
                     "test samples": sum(len(x.starts) for x in PARTS[k, "a"][0] + PARTS[k, "a"][1])}
                    for k, p in enumerate(POP)]).to_string(index=False))

## fig3d-number-of-cells

> Decoding accuracy increased with the number of cells used.

Reading (a) of the claims file: the 620-cell session, k = 1, 5, 10, 20, 30, 50, 100, 200, 300, 400 and
500, 100 random subsets each, whole-arena decoder. Reading (b), the four datasets pooled, is run
only if (a) does not meet the criterion under every speed and rate-map reading.

In [ ]:
K3 = [1, 5, 10, 20, 30, 50, 100, 200, 300, 400, 500]
rows = []
summ = []
t0 = time.time()
for sp, rm in READINGS:
    rng = np.random.default_rng(oc.seed_for("fig3d-number-of-cells", sp, rm))
    accs, ks = [], []
    for k in K3:
        for _ in range(100):
            units = list(rng.permutation(POP[BIG].n_units)[:k])
            accs.append(decode(BIG, sp, rm, units, "arena")[0])
            ks.append(k)
    accs, ks = np.array(accs), np.array(ks)
    r, p = stats.pearsonr(ks, accs)
    means = [accs[ks == k].mean() for k in K3]
    summ.append({"speed": sp, "rm": rm, "r": r, "p": p, "mean k=1": means[0], "mean k=500": means[-1],
                 "reproduced": r > 0 and p < 0.05 and means[-1] > means[0]})
    rows.append({"speed": sp, "rm": rm, **{f"k={k}": m for k, m in zip(K3, means)}})
print(pd.DataFrame(rows).to_string(index=False))
print(pd.DataFrame(summ).to_string(index=False))
print(f"({time.time() - t0:.0f} s)")
if not all(x["reproduced"] for x in summ):
    print("Reading (a) does not meet the criterion under every reading; reading (b) would be needed.")
else:
    print("Reading (a) meets the criterion under every reading; reading (b) is not needed for the outcome.")

**Result: reproduced.** On the 620-cell session, mean accuracy rises from 0.52 to 0.54 with one
cell to 0.74 to 0.77 with 500 cells. The Pearson correlation of accuracy with the number of cells over
the 1,100 subsets is r = 0.58 to 0.64, p < 1e-90, under every speed and rate-map reading. Reading (b)
is not needed.

## fig6c-subsets-decline and fig6d-reorganization-grows-with-cells

> In the 620-cell session, random subsets of every size except a single cell showed decoding accuracy
> falling with distance.

> The difference between bins 1 and 5 grew, and the slope of accuracy over distance became more
> negative, as more cells were used, both reaching their extremes at 500 cells.

20 random subsets per size of the 620 cells, distance decoder. Sizes: the union of the claims file's
three lists, each list evaluated: (a) 1, 5, 10, 30, 50, 100, 200, 300, 400, 500; (b) 1, 5, 20, 30, 50
to 500; (c) 1, 5, 10, 30, 100 to 500. Tests: (d) signed-rank of bin 1 against bin 5 over the 20
subsets; (e) Kruskal-Wallis across the 5 bins over the 20 subsets.

In [ ]:
SIZES = [1, 5, 10, 20, 30, 50, 100, 200, 300, 400, 500]
LISTS = {"a": [1, 5, 10, 30, 50, 100, 200, 300, 400, 500], "b": [1, 5, 20, 30, 50, 100, 200, 300, 400, 500],
         "c": [1, 5, 10, 30, 100, 200, 300, 400, 500]}
SUB = {}
t0 = time.time()
for sp, rm in READINGS:
    rng = np.random.default_rng(oc.seed_for("fig6c-subsets-decline", sp, rm))
    for n in SIZES:
        SUB[sp, rm, n] = np.array([decode(BIG, sp, rm, list(rng.permutation(POP[BIG].n_units)[:n]), "distance") for _ in range(20)])
print(f"({time.time() - t0:.0f} s)")
rows = []
for sp, rm in READINGS:
    for n in SIZES:
        A = SUB[sp, rm, n]
        w, pd_ = oc.signrank(A[:, 0], A[:, 4])
        h, pe = oc.kruskal(A.T)
        rows.append({"speed": sp, "rm": rm, "size": n, "mean bin1": A[:, 0].mean(), "mean bin5": A[:, 4].mean(),
                     "signed-rank p": pd_, "KW H": h, "KW p": pe, "mean diff": np.mean(A[:, 0] - A[:, 4]),
                     "mean slope": np.mean([oc.slope(a) for a in A])})
f6 = pd.DataFrame(rows)
print(f6.to_string(index=False))
out = []
for sp, rm in READINGS:
    d = f6[(f6.speed == sp) & (f6.rm == rm)].set_index("size")
    for lst, sizes in LISTS.items():
        for test, col in (("d", "signed-rank p"), ("e", "KW p")):
            single = d.loc[1, col] >= 0.05
            others = all(d.loc[n, col] < 0.05 and d.loc[n, "mean bin1"] > d.loc[n, "mean bin5"] for n in sizes if n > 1)
            out.append({"speed": sp, "rm": rm, "sizes": lst, "test": test, "single cell p >= 0.05": single,
                        "all others p < 0.05, bin 1 higher": others, "met": single and others})
f6c = pd.DataFrame(out)
print("\nfig6c criterion per reading:")
print(f6c.to_string(index=False))
print("\nPer speed, rm and size list, met under one test reading:")
print(f6c.groupby(["speed", "rm", "sizes"]).met.any().to_string())

rows = []
for sp, rm in READINGS:
    for lst, sizes in LISTS.items():
        diffs = [SUB[sp, rm, n][:, 0] - SUB[sp, rm, n][:, 4] for n in sizes]
        slopes = [np.array([oc.slope(a) for a in SUB[sp, rm, n]]) for n in sizes]
        h1, p1 = oc.kruskal(diffs)
        h2, p2 = oc.kruskal(slopes)
        md, ms = [x.mean() for x in diffs], [x.mean() for x in slopes]
        rows.append({"speed": sp, "rm": rm, "sizes": lst, "H diff": h1, "p diff": p1, "H slope": h2, "p slope": p2,
                     "size of largest mean diff": sizes[int(np.argmax(md))], "size of most negative slope": sizes[int(np.argmin(ms))],
                     "met": p1 < 0.05 and p2 < 0.05 and sizes[int(np.argmax(md))] == 500 and sizes[int(np.argmin(ms))] == 500})
f6d = pd.DataFrame(rows)
print("\nfig6d criterion per reading:")
print(f6d.to_string(index=False))

**Result: blocked (methods underspecified)** (fig6c-subsets-decline). The readings give different
outcomes. The single-cell curve shows no decline under every reading (signed-rank p = 0.23 to 0.76,
Kruskal-Wallis p ≥ 0.74). Every size from 100 cells up declines, with both tests p ≤ 0.0006. Small
subsets are mixed: under speed b with the legend's sizes (b) and the signed-rank test, every size of
that list from 5 cells up gives p < 0.05 with bin 1 higher, so the criterion is met. Under speed a, sizes of 5
to 30 cells give signed-rank p up to 0.35, and the criterion is met under no size list.

**Result: blocked (methods underspecified)** (fig6d-reorganization-grows-with-cells). The
Kruskal-Wallis tests across sizes give p < 0.05 for the difference under every reading. For the
slope they give p < 0.05 except under speed a, `rm` b (p = 0.055 to 0.10). Under speed b the largest
mean difference and the most negative mean slope are both at 500 cells, under every size list, so
the criterion is met. Under speed a the most negative slope is at 400 cells, and under speed a, `rm`
b the largest difference is too. The speed readings give different outcomes.

## figs8-smaller-datasets

> The growth of distance reorganization with the number of cells held in the tetrode sessions and in
> the 135-cell Neuropixels session.

20 subsets per size; sizes of reading (a) of fig6c up to each session's count, then the session's
full count. Tetrode sessions pooled (14 sessions x 20 subsets per size, the full count as one size
"max"). Kruskal-Wallis across sizes of the bin 1 minus bin 5 difference and of the slope.

In [ ]:
rows = []
t0 = time.time()
for sp, rm in READINGS:
    for name, ks in (("tetrode", TET), ("135-cell", [MID])):
        groups_d, groups_s = {}, {}
        for k in ks:
            n_all = POP[k].n_units
            sizes = [n for n in LISTS["a"] if n < n_all] + [n_all]
            rng = np.random.default_rng(oc.seed_for("figs8-smaller-datasets", sp, rm, k))
            for n in sizes:
                lab = "max" if n == n_all else n
                for _ in range(20):
                    acc = decode(k, sp, rm, list(rng.permutation(n_all)[:n]), "distance")
                    groups_d.setdefault(lab, []).append(acc[0] - acc[4])
                    groups_s.setdefault(lab, []).append(oc.slope(acc))
        labs = [l for l in LISTS["a"] if l in groups_d] + ["max"]
        h1, p1 = oc.kruskal([groups_d[l] for l in labs])
        h2, p2 = oc.kruskal([groups_s[l] for l in labs])
        md = [np.nanmean(groups_d[l]) for l in labs]
        ms = [np.nanmean(groups_s[l]) for l in labs]
        rows.append({"speed": sp, "rm": rm, "dataset": name, "sizes": " ".join(map(str, labs)),
                     "mean diff by size": " ".join(f"{x:.3f}" for x in md), "mean slope by size": " ".join(f"{x:.4f}" for x in ms),
                     "H diff": h1, "p diff": p1, "H slope": h2, "p slope": p2,
                     "diff rises": md[-1] > md[0] and stats.spearmanr(range(len(md)), md)[0] > 0,
                     "slope falls": ms[-1] < ms[0] and stats.spearmanr(range(len(ms)), ms)[0] < 0})
s8 = pd.DataFrame(rows)
s8["met"] = (s8["p diff"] < 0.05) & (s8["p slope"] < 0.05) & s8["diff rises"] & s8["slope falls"]
print(s8.to_string(index=False))
print(f"({time.time() - t0:.0f} s); total {time.time() - started:.0f} s")

**Result: reproduced.** In the 14 tetrode sessions pooled and in the 135-cell session, both
Kruskal-Wallis tests across sizes give p < 1e-15 under every reading (H = 480 to 573 tetrode, 84 to
100 for the 135-cell session). The mean difference rises and the mean slope falls with size. In the
tetrode sessions the full count (25 to 39 cells) is close to 30 cells (0.18 to 0.19 against 0.19 to
0.20).

## Summary

| Claim | Outcome |
|---|---|
| fig3d-number-of-cells | reproduced |
| fig6c-subsets-decline | blocked, methods underspecified (speed and size-list readings differ) |
| fig6d-reorganization-grows-with-cells | blocked, methods underspecified (speed readings differ) |
| figs8-smaller-datasets | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the Neuropixels sessions' tracking comes from the CA1
file; for the tetrode sessions the sizes beyond the session's count are replaced by the full count, as
the claims file says.

What was found about the dataset: nothing beyond notebook 01 (the y coordinate of the Neuropixels
population files is a copy of x).